# ExpectedSARSA 算法（期望 SARSA）

> Expected-SARSA 是 **SARSA 的方差缩减改进版**，同属 On-Policy TD 控制，核心区别在于：将更新目标中的单次采样 $Q(s',a')$ 替换为**对下一状态 $s'$ 上所有动作的期望 Q 值**。
>
> **普通 SARSA** 的更新公式（目标依赖实际采样到的 $a'$）：
>
> $$Q(s_t,a_t) \leftarrow Q(s_t,a_t) - \alpha \underbrace{\bigl[Q(s_t,a_t) - (r_{t+1} + \gamma Q(s_{t+1},a_{t+1}))\bigr]}_{\text{TD 误差，用 }(s,a,r,s',a')}$$
>
> **Expected-SARSA** 的更新公式（目标改为期望，消除 $a'$ 的采样噪声）：
>
> $$Q(s_t,a_t) \leftarrow Q(s_t,a_t) - \alpha \left[Q(s_t,a_t) - \left(r_{t+1} + \gamma \sum_{a'} \pi(a'|s_{t+1})\, Q(s_{t+1},a')\right)\right]
>$$
>
> 两者的核心对比如下：
>
> ---

| 对比维度 | 普通 SARSA | Expected SARSA |
|---|---|---|
| 更新目标 | 单次采样的 $Q(s',a')$ | $\sum_{a'}\pi(a'\|s')Q(s',a')$ 加权期望 |
| **方差** | **高**（随 $a'$ 随机波动） | **低**（期望操作消除采样噪声） |
| 偏差 | 无额外偏差 | 无额外偏差 |
| 收敛速度 | 较慢 | **更快**，尤其在随机策略阶段 |
| 每步计算量 | $O(1)$，仅用一个 $Q(s',a')$ | $O(\|\mathcal{A}\|)$，需遍历所有动作 |
| 策略类型 | On-policy | **On-policy / Off-policy 均可** |

> ---
>
> **总结**：Expected SARSA 用遍历动作空间的少量额外计算（对 GridWorld 仅 5 个动作，代价极低），换取更低的方差与更快的收敛速度，是对 SARSA 的直接升级。

## 一、导入库与环境初始化

In [1]:
# 导入 NumPy 库，用于数组操作（Q 表、策略矩阵等）
import numpy as np
# 导入 random 模块，用于随机选择初始状态和动作
import random
# 导入 importlib 标准库，用于按文件路径动态加载模块
import importlib.util
# 导入 os 模块，用于拼接当前目录与文件名
import os
# 导入 time 模块（备用，可控制训练展示节奏）
import time
# 导入 Jupyter 显示控制函数，训练时清除输出以实时刷新展示
from IPython.display import clear_output

# 环境文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头，
# Python 无法直接 import，需通过 importlib 按路径加载
_env_path = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "02.1.ModelFree_Env_GridWorldV2.py")  # 环境文件的绝对路径（str）
_spec = importlib.util.spec_from_file_location(
    "GridWorld_v2", _env_path)   # 构造模块规格对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)  # 根据规格创建模块对象
_spec.loader.exec_module(GridWorld_v2)                  # 执行模块代码，完成加载

In [ ]:
# 折扣因子（越接近 0 越短视，越接近 1 越长远）
gamma = 0.9

# 设置网格世界行数（必须与 desc 字符串行数一致）
rows = 5
# 设置网格世界列数（必须与 desc 每行字符数一致）
columns = 5

# 创建 5×5 GridWorld 环境实例
# forbiddenAreaScore=-10: 进入障碍格（'#'）的即时惩罚
# score=1: 到达目标格（'T'）的即时正奖励
# desc: 地图描述，'.'=普通格，'#'=障碍格，'T'=目标格
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10, score=1,
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."])

# 显示网格地图
gridworld.show()

# 初始化状态价值向量（全零，用于展示）
# shape: (25,)——每个元素对应一个状态（0~24）的价值估计
value = np.zeros(rows * columns)

# 初始化 Q 表（动作价值矩阵，全零）
# shape: (25, 5)——行=状态索引[0~24]，列=5种动作的 Q 值
qtable = np.zeros((rows * columns, 5))

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️


## 二、Expected-SARSA 算法实现

**核心思想**：On-Policy TD 控制（期望版本）

Q 值更新公式（与 SARSA 的关键区别）：
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) - \alpha \left[Q(s_t, a_t) - \left(r_{t+1} + \gamma \sum_{a'}\pi(a'|s_{t+1}) Q(s_{t+1}, a')\right)\right]$$

Expected-SARSA 用**期望 Q 值**（对下一状态所有动作加权平均）代替 SARSA 中的单个 $Q(s',a')$，方差更小、更稳定。

In [ ]:
def Expected_SARSA(gridworld: GridWorld_v2.GridWorld_v2,
                   gamma=0.99,
                   trajectorySteps=-1,
                   learning_rate=0.001,
                   final_epsilon=0.01,
                   num_episodes=600) -> GridWorld_v2.GridWorld_v2:
    '''
    Expected-SARSA 算法（期望 SARSA）。

    在 SARSA 基础上，将更新目标中的单步 Q(s',a') 替换为
    下一状态动作价值的期望值 E_π[Q(s',·)]，从而降低方差，
    加快收敛速度，通常优于标准 SARSA。

    参数:
        gridworld (GridWorld_v2): GridWorld 环境实例
        gamma (float): 折扣因子，衡量未来奖励的重要程度，取值 (0,1]
        trajectorySteps (int): 每条轨迹最大步数；-1 表示到目标自动停止
        learning_rate (float): TD 学习率，控制 Q 值更新幅度
        final_epsilon (float): epsilon 的最小值，防止完全停止探索
        num_episodes (int): 训练轮数

    返回值:
        GridWorld_v2.GridWorld_v2: 训练后的环境对象（可查看最终策略）
    '''
    # 初始化状态价值向量（全零，仅用于计算展示）
    # shape: (25,)
    state_value = np.zeros((rows * columns))

    # 初始化动作价值矩阵 Q（全零）
    # shape: (25, 5)——行=状态[0~24]，列=5种动作的 Q 值
    action_value = np.zeros((rows * columns, 5))

    # 初始化随机确定性策略（独热编码）
    # shape: (25, 5)——每行只有一个位置为 1（对应随机选取的初始动作）
    policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]

    epsilon = 0.5  # 初始 epsilon（较高，鼓励早期探索）

    for episode in range(num_episodes):  # 训练主循环
        print("episode", f"{episode}/{num_episodes}")  # 打印当前轮次

        # 线性衰减 epsilon
        if epsilon > final_epsilon:
            epsilon -= 0.001   # 每轮减少 0.001
        else:
            epsilon = final_epsilon  # 不低于最小探索率

        # 计算 epsilon-greedy 策略的概率分配
        # p1：贪心动作的选取概率 = 1 - ε*(4/5)
        p1 = 1 - epsilon * (4/5)
        # p0：非贪心动作各自的选取概率 = ε/5
        p0 = epsilon / 5
        # 概率映射：策略矩阵中 1→p1，0→p0
        d = {1: p1, 0: p0}
        print("p1", p1, "p0", p0)  # 打印当前概率值

        # 将确定性策略转为 epsilon-greedy 概率分布
        # shape: (25, 5)——每行是一个状态的动作概率分布
        policy_epsilon = np.vectorize(d.get)(policy)

        # 初始化状态访问计数器（长度 25）
        cnt = [0 for i in range(25)]

        # 固定初始状态为 10（地图中部）
        initState = 10
        # 随机选择初始动作
        initAction = random.randint(0, 4)

        if trajectorySteps == -1:
            stop_when_reach_target = True  # 到达目标时自动终止轨迹

        # 按 epsilon-greedy 策略生成轨迹
        # 返回值：列表，每个元素为 (状态, 动作, 奖励, 下一状态, 下一动作) 五元组
        Trajectory = gridworld.getTrajectoryScore(
            nowState=initState,
            action=initAction,
            policy=policy_epsilon,
            steps=trajectorySteps,
            stop_when_reach_target=True)

        # 在轨迹末尾追加目标格自循环，稳定终止状态的 Q 值更新
        Trajectory.append((17, 4, 1, 17, 4))
        print("trajectorySteps", len(Trajectory))  # 打印轨迹长度

        steps = len(Trajectory) - 1  # 有效步数

        # 从轨迹末尾向前遍历，执行 Expected-SARSA TD 更新
        for k in range(steps, -1, -1):
            # 解包当前步转移
            # tmpstate:   当前状态 int
            # tmpaction:  当前动作 int
            # tmpscore:   即时奖励 float
            # nextState:  下一状态 int
            # nextAction: 下一动作 int（Expected-SARSA 中不直接使用）
            tmpstate, tmpaction, tmpscore, nextState, nextAction = Trajectory[k]
            cnt[tmpstate] += 1  # 记录状态访问次数

            # 计算下一状态的期望 Q 值（Expected-SARSA 核心）
            # action_value[nextState]: shape (5,)——nextState 下 5 种动作的 Q 值
            # policy_epsilon[nextState]: shape (5,)——nextState 下各动作的选取概率
            # v = Σ_a π(a|s') * Q(s', a) ——加权期望，标量 float
            v = (action_value[nextState] * policy_epsilon[nextState]).sum()

            # TD 误差：Q(s,a) - [r + γ * E_π[Q(s',·)]]
            # 用期望值代替 SARSA 中的单个 Q(s',a')，方差更小
            TD_error = action_value[tmpstate][tmpaction] - (tmpscore + gamma * v)
            # 按 TD 误差更新 Q 值
            action_value[tmpstate][tmpaction] -= learning_rate * TD_error

        # 策略改进：每个状态选 Q 值最大的动作
        # np.argmax(action_value, axis=1)：shape (25,)，每个状态的最优动作索引
        policy = np.eye(5)[np.argmax(action_value, axis=1)]  # shape (25, 5)
        # 更新 epsilon-greedy 策略
        policy_epsilon = np.vectorize(d.get)(policy)

        print(np.array(cnt).reshape(5, 5))  # 打印 5×5 状态访问次数矩阵

        # 计算状态价值（加权期望 Q 值）用于展示
        # V(s) = Σ_a π(a|s)*Q(s,a)，shape (25,)
        state_value = np.sum(policy_epsilon * action_value, axis=1)
        # 全局平均状态价值
        mean_state_value = np.sum(policy_epsilon * action_value, axis=1).mean()

        gridworld.showPolicy(policy)                             # 显示当前策略
        print(np.round(state_value, decimals=4).reshape(5, 5))  # 打印 5×5 状态价值
        print("mean_state_value", mean_state_value)             # 打印平均价值

    return gridworld  # 返回训练后的环境对象

## 三、运行 Expected-SARSA 算法并查看结果

In [ ]:
# 调用 Expected-SARSA 算法进行训练
# 效果通常优于 SARSA：收敛更稳定，不易出现局部抖动，可达到全局最优
Expected_SARSA(gridworld)

episode 0/600
p1 0.6008 p0 0.0998
trajectorySteps 56
[[26  1  1  0  0]
 [ 5  0  1  0  0]
 [ 2  0  1  2  4]
 [ 0  1  2  1  3]
 [ 0  1  1  1  3]]
➡️⬆️⬆️⬆️⬆️
➡️⏫️⏫️⬆️⬆️
➡️⬆️⏫️⬆️⬆️
⬆️⏩️✅⏫️⬆️
⬆️⏩️⬆️⬆️⬇️
[[-0.0015 -0.     -0.001   0.      0.    ]
 [-0.      0.     -0.001   0.      0.    ]
 [-0.      0.     -0.     -0.     -0.0001]
 [ 0.      0.0006 -0.0004 -0.     -0.001 ]
 [ 0.     -0.001  -0.001  -0.     -0.0001]]
mean_state_value -0.00025958809591138473
episode 1/600
p1 0.6015999999999999 p0 0.0996
trajectorySteps 1521
[[ 41 264 231 319 428]
 [ 11  54  43  42  57]
 [  3   8   4   2   9]
 [  1   0   2   1   1]
 [  0   0   0   0   0]]
🔄⬅️🔄⬇️⬇️
⬅️⏪⏩️⬇️⬇️
⬅️⬇️⏬⬆️⬅️
➡️⏩️✅⏫️➡️
⬆️⏩️⬆️⬆️⬇️
[[-0.0024 -0.0451 -0.0398 -0.0184 -0.0305]
 [-0.007  -0.0112 -0.0111 -0.0061 -0.0009]
 [-0.     -0.008  -0.0024 -0.001  -0.0001]
 [-0.      0.0006  0.0002 -0.     -0.001 ]
 [ 0.     -0.001  -0.001  -0.     -0.0001]]
mean_state_value -0.007445822371918201
episode 2/600
p1 0.6024 p0 0.0994
trajectorySteps 35
[[ 0